In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random


In [ ]:
def foveated_warp(img, center, output_size=224):
    """
    img: 1024x1024 input
    center: (x, y) of your 30-40px ROI
    """
    # Create a coordinate map
    xs = np.linspace(-1, 1, output_size)
    ys = np.linspace(-1, 1, output_size)
    xv, yv = np.meshgrid(xs, ys)

    # Apply a power-law distortion (The 'Squeeze')
    # An exponent > 1 pulls more of the outer image into the frame
    exponent = 1.5
    dist_x = np.sign(xv) * np.power(np.abs(xv), exponent)
    dist_y = np.sign(yv) * np.power(np.abs(yv), exponent)

    # Map back to original image coordinates
    # (Simplified: assumes center is the focus)
    map_x = ((dist_x + 1) / 2 * 1024).astype(np.float32)
    map_y = ((dist_y + 1) / 2 * 1024).astype(np.float32)

    # Remap the image
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

In [ ]:
img = plt.imread(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/other/ebc0c1ca-8819-49f3-a989-0cb5b3db66ec_87/image.jpg"
)
plt.imshow(img)

In [ ]:
plt.imshow(foveated_warp(img, (200, 250), 224))

I basically want to define the amount of interpolation happening at each point.  
It is very useful to have opencv do interpolation for me, but oh well.  

Technically, the easiest way to go about this is to interpolate aggressively outside roi, and put it behind the roi. although that would be quite weird.  
or i define the amount of interpolation as a gaussian curve starting from roi.  

In [ ]:
import cv2
import numpy as np


def gaussian_variable_res(img, center, output_sz=224, sigma=60, amplitude=7.0):
    """
    img: 1024x1024 input
    center: (cx, cy) of ROI
    sigma: Controls how 'wide' the high-res area is.
           Lower sigma = faster drop-off to low-res.
    """
    cx, cy = center

    # 1. Create a 1D axis for the output image
    ax = np.linspace(-(output_sz // 2), output_sz // 2, output_sz)

    # 2. Define the 'Step Size' function (Inverted Gaussian)
    # At the center (0), step is 1.0 (1:1 resolution)
    # At the edges, step is > 1.0 (downsampling/skipping pixels)
    # We use: step(d) = 1 + Amplitude * (1 - exp(-d^2 / 2*sigma^2))
    # Let's say we want the edges to be 8x downsampled

    def get_map_axis(axis_range, center_val, max_source):
        # Calculate steps from center outward
        dist = np.abs(axis_range)
        steps = 1 + amplitude * (1 - np.exp(-(dist**2) / (2 * sigma**2)))

        # Cumulative sum gives the source coordinates
        # We center it so that the 0 in our axis_range maps to center_val
        pos = np.cumsum(steps)
        pos -= pos[output_sz // 2]  # Re-center the middle of the array to 0
        return pos + center_val

    # 3. Generate X and Y maps
    map_x_1d = get_map_axis(ax, cx, img.shape[1])
    map_y_1d = get_map_axis(ax, cy, img.shape[0])

    # Create the 2D meshgrid for cv2.remap
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # 4. Perform the Interpolation
    # INTER_LINEAR is bilinear interpolation
    foveated = cv2.remap(
        img.astype(np.float32),
        map_x.astype(np.float32),
        map_y.astype(np.float32),
        interpolation=cv2.INTER_AREA,
    )

    return foveated.astype(np.uint8)

In [ ]:
def fn(x):
    return 1 + 7 * (1 - np.exp(-(x**2) / (2 * 60 * 60)))

In [ ]:
xs = np.linspace(-112, 112, 224)
ys = fn(xs)
plt.plot(xs, ys)
plt.show()

In [ ]:
np.linspace(-(224 // 2), 224 // 2, 224)

In [ ]:
img = plt.imread(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall/3958731314210636/image.jpg"
)
plt.imshow(img)

In [ ]:
plt.imshow(gaussian_variable_res(img, (300, 300), 224, amplitude=5))

In [ ]:
import cv2
import numpy as np


def get_full_context_map(source_size, output_size, roi_coord, sigma=30):
    """
    source_size: 1024
    output_size: 224
    roi_coord: The x or y center of your interest
    """
    # 1. Create a relative axis from -112 to 111
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate raw 'weights' (Inverted Gaussian)
    # Center = 1.0 (High Res). Edges = High value (Low Res/Squeezed)
    # We use (1 - G) to make the center the 'slowest' part of the map
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(out_axis**2) / (2 * sigma**2)))

    # 3. Split weights into two halves based on the ROI position
    # This ensures the ROI stays at the correct physical location
    left_weights = weights[: output_size // 2][::-1]  # Weights from ROI to 0
    right_weights = weights[output_size // 2 :]  # Weights from ROI to 1023

    # 4. Scale weights so the cumulative sum matches the distance to edges
    left_coords = roi_coord - np.cumsum(left_weights) * (
        roi_coord / np.sum(left_weights)
    )
    right_coords = roi_coord + np.cumsum(right_weights) * (
        (source_size - 1 - roi_coord) / np.sum(right_weights)
    )

    # Combine and sort to get the full mapping for this axis
    return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


def warp_no_crop(img, roi_center):
    cx, cy = roi_center

    # Generate maps for both axes
    map_x_1d = get_full_context_map(1024, 224, cx)
    map_y_1d = get_full_context_map(1024, 224, cy)

    # Create the 2D grid
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # Remap - this pulls the WHOLE 1024 image into the 224 canvas
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

In [ ]:
plt.imshow(warp_no_crop(img, (500, 350)))

# cv2.remap

This seems like a useful function. The next goal is to create a focused image which progressively downsizes on the edges, but keeps the roi in the same resolution (in the last stage, we run a final resize too though).  

cv2.remap takes mapping arrays, which at coordinate (x,y) give the coordinate in the source image for the destination image.  
So we kinda do the interpolation ourselves.  

We would basically take the roi, map it one-to-one in the mapping array. Determine the size of the total array from the roi + pad amount.  
For the pad amount, we progressively resize it (it shrinks at the edges).  

I need to learn to use it, lets do a translate first manually

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
img = plt.imread(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall/3958731314210636/image.jpg"
)
plt.imshow(img)

In [ ]:
# translate the image now
# we create the same image, but each x coordinate is shifted by 100px

map_x = np.zeros(img.shape[:2], dtype=np.float32)
map_y = np.zeros(img.shape[:2], dtype=np.float32)

for r in range(img.shape[0]):
    for c in range(img.shape[1]):
        map_x[r][c] = np.clip(c - 100, 0, img.shape[1])
        map_y[r][c] = r

In [ ]:
# id map_x


def _get_id_map_x(img):
    map_x = np.stack([np.arange(img.shape[1], dtype=np.float32)] * img.shape[0])
    return map_x


def _get_id_map_y(img):
    map_y = np.stack([np.arange(img.shape[0], dtype=np.float32)] * img.shape[1]).T
    return map_y

In [ ]:
# id remapping
map_x = _get_id_map_x(img)
map_y = _get_id_map_y(img)
remapped = cv2.remap(img, map_x, map_y, cv2.INTER_AREA)
show([remapped, img])

In [ ]:
# translate x by 100
# translate y by 250
map_x = _get_id_map_x(img)
map_y = _get_id_map_y(img)
map_x = np.clip(map_x - 100, 0, img.shape[1])
map_y = np.clip(map_y - 250, 0, img.shape[0])
remapped = cv2.remap(img, map_x, map_y, cv2.INTER_AREA)
show([img, remapped])

In [ ]:
# you can get 0 padding by first padding the original image by 0pad

pad_width_rgb = ((1, 1), (1, 1), (0, 0))
padded = np.pad(img, pad_width=pad_width_rgb, mode="constant", constant_values=0)
print(padded.shape)
map_x = _get_id_map_x(padded)
map_y = _get_id_map_y(padded)
map_x = np.clip(map_x - 100, 0, padded.shape[1])
map_y = np.clip(map_y - 250, 0, padded.shape[0])
remapped = cv2.remap(padded, map_x, map_y, cv2.INTER_AREA)
show([img, remapped])

lets do a flip now

The current pixel is the shape - pixel-coord, or simply, the reverse of map_x

In [ ]:
np.arange(10, -10, -1)

In [ ]:
from mtrain.utils import *


# only x flipping
def _get_id_map_x(img):
    map_x = np.stack([np.arange(img.shape[1], dtype=np.float32)] * img.shape[0])
    return map_x


def _get_id_map_y(img):
    map_y = np.stack([np.arange(img.shape[0], dtype=np.float32)] * img.shape[1]).T
    return map_y


map_x = np.stack([np.arange(img.shape[1], 0, -1, dtype=np.float32)] * img.shape[0])
map_y = _get_id_map_y(img)
remapped = cv2.remap(img, map_x, map_y, cv2.INTER_AREA)
show([img, remapped])

In [ ]:
def get_full_context_map(source_size, output_size, roi_coord, sigma=30):
    """
    source_size: 1024
    output_size: 224
    roi_coord: The x or y center of your interest
    """
    # 1. Create a relative axis from -112 to 111
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate raw 'weights' (Inverted Gaussian)
    # Center = 1.0 (High Res). Edges = High value (Low Res/Squeezed)
    # We use (1 - G) to make the center the 'slowest' part of the map
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(out_axis**2) / (2 * sigma**2)))

    # 3. Split weights into two halves based on the ROI position
    # This ensures the ROI stays at the correct physical location
    left_weights = weights[: output_size // 2][::-1]  # Weights from ROI to 0
    right_weights = weights[output_size // 2 :]  # Weights from ROI to 1023

    # 4. Scale weights so the cumulative sum matches the distance to edges
    left_coords = roi_coord - np.cumsum(left_weights) * (
        roi_coord / np.sum(left_weights)
    )
    right_coords = roi_coord + np.cumsum(right_weights) * (
        (source_size - 1 - roi_coord) / np.sum(right_weights)
    )

    # Combine and sort to get the full mapping for this axis
    return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


def warp_no_crop(img, roi_center):
    cx, cy = roi_center

    # Generate maps for both axes
    map_x_1d = get_full_context_map(1024, 224, cx)
    map_y_1d = get_full_context_map(1024, 224, cy)

    # Create the 2D grid
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # Remap - this pulls the WHOLE 1024 image into the 224 canvas
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

In [ ]:
import cv2
import numpy as np


def visualize_warp_grid(roi_center, source_size=1024, output_size=224):
    # 1. Create a blank 'grid' image
    grid_img = np.full((source_size, source_size), 255, dtype=np.uint8)
    for i in range(0, source_size, 32):
        cv2.line(grid_img, (i, 0), (i, source_size), (0, 0, 0), 1)
        cv2.line(grid_img, (0, i), (source_size, i), (0, 0, 0), 1)

    # 2. Generate the "No-Crop" Maps (using the function from before)
    map_x_1d = get_full_context_map(source_size, output_size, roi_center[0])
    map_y_1d = get_full_context_map(source_size, output_size, roi_center[1])
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # 3. Warp the grid
    warped_grid = cv2.remap(grid_img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

    return grid_img, warped_grid

In [ ]:
import albumentations as A
import cv2
import numpy as np


def get_full_context_map(source_size, output_size, roi_coord, sigma=30):
    """
    source_size: 1024
    output_size: 224
    roi_coord: The x or y center of your interest
    """
    # 1. Create a relative axis from -112 to 111
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate raw 'weights' (Inverted Gaussian)
    # Center = 1.0 (High Res). Edges = High value (Low Res/Squeezed)
    # We use (1 - G) to make the center the 'slowest' part of the map
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(out_axis**2) / (2 * sigma**2)))

    # 3. Split weights into two halves based on the ROI position
    # This ensures the ROI stays at the correct physical location
    left_weights = weights[: output_size // 2][::-1]  # Weights from ROI to 0
    right_weights = weights[output_size // 2 :]  # Weights from ROI to 1023

    # 4. Scale weights so the cumulative sum matches the distance to edges
    left_coords = roi_coord - np.cumsum(left_weights) * (
        roi_coord / np.sum(left_weights)
    )
    right_coords = roi_coord + np.cumsum(right_weights) * (
        (source_size - 1 - roi_coord) / np.sum(right_weights)
    )

    # Combine and sort to get the full mapping for this axis
    return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


def warp_no_crop(img, roi_center):
    tfm = A.Compose(
        [
            A.Resize(1024, 1024, cv2.INTER_AREA),
            A.PadIfNeeded(1024, 1024),
        ]
    )
    img = tfm(image=img)["image"]

    cx, cy = roi_center

    # Generate maps for both axes
    map_x_1d = get_full_context_map(1024, 224, cx)
    map_y_1d = get_full_context_map(1024, 224, cy)

    # Create the 2D grid
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # Remap - this pulls the WHOLE 1024 image into the 224 canvas
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

In [ ]:
plt.imshow(img)

In [ ]:
plt.imshow(warp_no_crop(img, (300, 350)))

gemini function description

- create arange coords starting from -size/2 to size/2
    - this centers 0 at the center
- until now, we dont have an roi coord
- run them through a function (inverted gaussian). This gievs this func value at each coord
- split the list in two halves, the left half is reversed
  - due to the reversal both lists are monotonically increasing from 1

In [ ]:
output_size = 64
sigma = 60
source_size = 640
roi_coord = 320

# 1. Create a relative axis from -112 to 111
out_axis = np.arange(output_size) - (output_size // 2)
out_axis

In [ ]:
# 2. Calculate raw 'weights' (Inverted Gaussian)
# Center = 1.0 (High Res). Edges = High value (Low Res/Squeezed)
# We use (1 - G) to make the center the 'slowest' part of the map
weights = 1.0 + 10.0 * (1.0 - np.exp(-(out_axis**2) / (2 * sigma**2)))
weights

In [ ]:
# 3. Split weights into two halves based on the ROI position
# This ensures the ROI stays at the correct physical location
left_weights = np.flip(weights[: output_size // 2])  # Weights from ROI to 0
right_weights = weights[output_size // 2 :]  # Weights from ROI to 1023

In [ ]:
_, axes = plt.subplots(1, 3)
axes[0].plot(weights)
axes[1].plot(left_weights)
axes[2].plot(right_weights)
plt.show()

In [ ]:
plt.plot(np.cumsum(left_weights))

In [ ]:
# 4. Scale weights so the cumulative sum matches the distance to edges
left_coords = roi_coord - np.cumsum(left_weights) * (roi_coord / np.sum(left_weights))
right_coords = roi_coord + np.cumsum(right_weights) * (
    (source_size - 1 - roi_coord) / np.sum(right_weights)
)

# Combine and sort to get the full mapping for this axis

In [ ]:
_, ax = plt.subplots(1, 2)
ax[0].plot(np.flip(left_coords))
ax[1].plot(right_coords)
plt.show()

In [ ]:
left_coords

In [ ]:
map_val = np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)
plt.plot(map_val)
plt.show()

In [ ]:
plt.plot(get_full_context_map(1024, 224, 512))

In [ ]:
import numpy as np
import cv2
import albumentations as A
import warnings
from dataclasses import dataclass
from mtrain.neg_mask.crops import Bbox


def get_bbox_context_map(source_size, output_size, bbox_start, bbox_end, sigma=30):
    box_dim = bbox_end - bbox_start

    # --- Sanity Check ---
    if box_dim > output_size:
        warnings.warn(
            f"Bbox dimension ({box_dim}) is larger than output size ({output_size}). "
            f"The ROI will be downsampled to fit."
        )

    # 1. Create our target axis (the 224 pixels)
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate the 'Flat Zone'
    # We want a 1:1 mapping inside the box.
    # half_box_out is how many pixels in the 224-output represent the box.
    half_box_out = box_dim // 2

    # Distance is 0 inside the box, increasing Gaussian outside
    dist_from_box = np.maximum(0, np.abs(out_axis) - half_box_out)

    # 3. Create Weights
    # 1.0 (Native) inside the box, scaling up to 11.0 at the far edges
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(dist_from_box**2) / (2 * sigma**2)))

    # 4. Split and Normalize
    roi_center = (bbox_start + bbox_end) // 2

    left_weights = weights[: output_size // 2][::-1]
    right_weights = weights[output_size // 2 :]

    # Summing weights and scaling to hit 0 and source_size exactly
    left_coords = roi_center - np.cumsum(left_weights) * (
        roi_center / np.sum(left_weights)
    )
    right_coords = roi_center + np.cumsum(right_weights) * (
        (source_size - 1 - roi_center) / np.sum(right_weights)
    )

    return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


def warp_bbox_no_crop(
    img, bbox: Bbox, initial_image_size: int = 1024, downsampled_size: int = 224
):
    # Standardize input
    tfm = A.Compose(
        [
            A.Resize(initial_image_size, initial_image_size, cv2.INTER_AREA),
            A.PadIfNeeded(
                initial_image_size, initial_image_size, border_mode=cv2.BORDER_CONSTANT
            ),
        ]
    )
    img = tfm(image=img)["image"]

    # Generate maps
    map_x_1d = get_bbox_context_map(
        initial_image_size, downsampled_size, bbox.x, bbox.x2
    )
    map_y_1d = get_bbox_context_map(
        initial_image_size, downsampled_size, bbox.y, bbox.y2
    )

    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)


    # INTER_AREA is often better for shrinking, but LINEAR is faster for warps
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

In [ ]:
def get_bbox_context_map(source_size, output_size, bbox_start, bbox_end, sigma=30):
    box_dim = bbox_end - bbox_start

    # --- Sanity Check ---
    if box_dim > output_size:
        warnings.warn(
            f"Bbox dimension ({box_dim}) is larger than output size ({output_size}). "
            f"The ROI will be downsampled to fit."
        )

    # 1. Create our target axis
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate the 'Flat Zone'
    half_box_out = box_dim // 2
    dist_from_box = np.maximum(0, np.abs(out_axis) - half_box_out)

    # 3. Create Weights
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(dist_from_box**2) / (2 * sigma**2)))

    # 4. Split and Normalize
    roi_center = (bbox_start + bbox_end) // 2
    left_weights = weights[: output_size // 2][::-1]
    right_weights = weights[output_size // 2 :]

    left_coords = roi_center - np.cumsum(left_weights) * (
        roi_center / np.sum(left_weights)
    )
    right_coords = roi_center + np.cumsum(right_weights) * (
        (source_size - 1 - roi_center) / np.sum(right_weights)
    )

    return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


def warp_bbox_no_crop(
    img, bbox: Bbox, initial_image_size: int = 1024, downsampled_size: int = 224
):
    # Standardize input
    tfm = A.Compose(
        [
            A.Resize(initial_image_size, initial_image_size, cv2.INTER_AREA),
            A.PadIfNeeded(
                initial_image_size, initial_image_size, border_mode=cv2.BORDER_CONSTANT
            ),
        ]
    )
    img = tfm(image=img)["image"]

    # Generate 1D maps
    map_x_1d = get_bbox_context_map(
        initial_image_size, downsampled_size, bbox.x, bbox.x2
    )
    map_y_1d = get_bbox_context_map(
        initial_image_size, downsampled_size, bbox.y, bbox.y2
    )

    # --- Calculate New Bbox Coordinates ---
    # np.searchsorted finds where the original coordinates fall in our non-linear 1D maps
    new_x = np.searchsorted(map_x_1d, bbox.x)
    new_y = np.searchsorted(map_y_1d, bbox.y)
    new_x2 = np.searchsorted(map_x_1d, bbox.x2)
    new_y2 = np.searchsorted(map_y_1d, bbox.y2)

    # Clip to output dimensions to be safe
    new_x, new_x2 = np.clip([new_x, new_x2], 0, downsampled_size - 1)
    new_y, new_y2 = np.clip([new_y, new_y2], 0, downsampled_size - 1)

    warped_bbox = Bbox(
        x=int(new_x),
        y=int(new_y),
        w=int(max(1, new_x2 - new_x)),
        h=int(max(1, new_y2 - new_y)),
    )

    # Create 2D meshgrid for remapping
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)
    warped_img = cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

    return warped_img, warped_bbox

In [ ]:
from mtrain.neg_mask.leveled_cropping import (
    load_crop_level_sample_from_directory,
    make_crop_level_pairs_v2,
)

sample = load_crop_level_sample_from_directory(
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/trash/393359396284943_24"
    )
)

In [ ]:
img, mask, bb = sample.full_image, sample.full_mask, sample.bbox

warped, _ = warp_bbox_no_crop(img, bb)
show([img, mask, warped])

AI models are not able to give me a good version which provides the bbox correctly in the output image.  

The basic algorithm does not change, we go outwards from the bbox in both directions and decide the step size using an inverted gaussian like function.   
I dont mind having a last final resize, so this is the strategy:
- do a cumsum of left separately
- cumsum of right separately
- left is up and right is down in y direction
- put the bbox in some location (which is generally the center, I guess padding at the edges is fine for now, although I need to decide if that is the right thing to do [we lose a lot of space in case the image is on the edge])

- Lets say we dont do center?
  - We have some ratios on the left and the right
  - we maintain those ratios in the output
  - this way we should be fine

In [ ]:
output_size = 224
# given the box coordinate, the output size and the input size
# we can find the left and right separetely

# these are top/bottom in y direction, for now do x
bb_dim = 40
in_size = 1024
left = 200
right = 250

# 0 -> 200 left portion
# 250 -> 1024 right portion
# we preserve these ratios in the output
# assuming the inner bbox is 40px atleast, we pad by 200
# we will do a final resize later

left_portion, right_portion = left, in_size - right
total = left_portion + right_portion
left_ratio, right_ratio = left_portion / total, right_portion / total

out_left_portion, out_right_portion = (
    int(left_ratio * output_size),
    int(right_ratio * output_size),
)
# now we need steps

left_range = np.arange(left_portion, 0, -1)
right_range = np.arange(right, right + right_portion, 1)

# left_range = np.arange(out_left_portion, 0, -1)
# start_right = out_left_portion + bb_dim
# right_range = np.arange(start_right, start_right + out_right_portion, 1)

left_range, right_range


# out_axis = np.arange(output_size) - (output_size // 2)
# out_axis

In [ ]:
def get_weights(dist_from_box, ampl, sigma):
    return 1.0 + ampl * (1.0 - np.exp(-(dist_from_box**2) / (2 * sigma**2)))

In [ ]:
left_weights = get_weights(left_range, 5, 60)
left_range, left_weights

In [ ]:
# take these many steps everytime and reach the bbox
left_coords = np.cumsum(left_weights)
bbox_start_from_left_coords = left_coords[-1]
actual_start = out_left_portion

((left_coords * actual_start) / bbox_start_from_left_coords).astype(np.uint16)

# last is 1954 -> the final coord
# we want it to be 200
# 1954 -> 200
# 11 -> ?
# 11 * (200 / 1954) = 0

# left_coords = roi_coord - np.cumsum(left_weights) * (
#     roi_coord / np.sum(left_weights)
# )

In [ ]:
out_axis = np.arange(output_size) - (output_size // 2)

# 2. Calculate the 'Flat Zone'
# We want a 1:1 mapping inside the box.
# half_box_out is how many pixels in the 224-output represent the box.
half_box_out = bb_dim // 2

# Distance is 0 inside the box, increasing Gaussian outside
dist_from_box = np.maximum(0, np.abs(out_axis) - half_box_out)

# 3. Create Weights
# 1.0 (Native) inside the box, scaling up to 11.0 at the far edges
weights = 1.0 + 10.0 * (1.0 - np.exp(-(dist_from_box**2) / (2 * sigma**2)))

# 4. Split and Normalize
roi_center = (left + right) // 2

left_weights = weights[: output_size // 2][::-1]
right_weights = weights[output_size // 2 :]

# Summing weights and scaling to hit 0 and source_size exactly
left_coords = roi_center - np.cumsum(left_weights) * (roi_center / np.sum(left_weights))
right_coords = roi_center + np.cumsum(right_weights) * (
    (source_size - 1 - roi_center) / np.sum(right_weights)
)

In [ ]:
left_coords[::-1]

In [ ]:
xs = np.arange(-100, 100)
ys = get_weights(xs, 8, 50)
plt.plot(xs, ys)

In [ ]:
def get_bbox_context_map(source_size, output_size, bbox_start, bbox_end, sigma=30):
    box_dim = bbox_end - bbox_start

    # --- Sanity Check ---
    if box_dim > output_size:
        warnings.warn(
            f"Bbox dimension ({box_dim}) is larger than output size ({output_size}). "
            f"The ROI will be downsampled to fit."
        )

    # 1. Create our target axis (the 224 pixels)
    out_axis = np.arange(output_size) - (output_size // 2)

    # 2. Calculate the 'Flat Zone'
    # We want a 1:1 mapping inside the box.
    # half_box_out is how many pixels in the 224-output represent the box.
    half_box_out = box_dim // 2

    # Distance is 0 inside the box, increasing Gaussian outside
    dist_from_box = np.maximum(0, np.abs(out_axis) - half_box_out)

    # 3. Create Weights
    # 1.0 (Native) inside the box, scaling up to 11.0 at the far edges
    weights = 1.0 + 10.0 * (1.0 - np.exp(-(dist_from_box**2) / (2 * sigma**2)))

    # 4. Split and Normalize
    roi_center = (bbox_start + bbox_end) // 2

    left_weights = weights[: output_size // 2][::-1]
    right_weights = weights[output_size // 2 :]

    # Summing weights and scaling to hit 0 and source_size exactly
    left_coords = roi_center - np.cumsum(left_weights) * (
        roi_center / np.sum(left_weights)
    )
    # right_coords = roi_center + np.cumsum(right_weights) * (
    #     (source_size - 1 - roi_center) / np.sum(right_weights)
    # )
    return left_coords.astype(np.float32)
    # return left_weights, right_weights, roi_center

    # return np.concatenate([left_coords[::-1], right_coords]).astype(np.float32)


In [ ]:
left_coords = get_bbox_context_map(1024, 224, 200, 250)
plt.plot(left_coords)

In [ ]:
def get_weights(dist, ampl, sigma):
    return 1.0 + ampl * (1.0 - np.exp(-(dist**2) / (2 * sigma**2)))


def get_scaled_array(orig_size, downsampled_size, ampl=10.0, sigma=20.0):
    # walk k (224) steps to cover n (1024) steps
    # we create an array of size downsampled_size which describes the points to take from 0 -> orig_size

    out_steps = np.arange(0, downsampled_size, 1)
    weights = get_weights(out_steps, ampl, sigma)
    total_weighted_steps = weights.sum()
    step_locs = weights.cumsum()

    return ((step_locs * orig_size) / total_weighted_steps).astype(np.float32)


def _get_downsampled_portions(bb_left, bb_right, length, desired_length):
    desired_left = int((bb_left * desired_length) / length)
    right_length = length - bb_right
    desired_right = int((right_length * desired_length) / length)
    return desired_left, desired_right


def _get_remap_array(left_point, right_point, original_length, desired_length, sigma):
    left_len, right_len = _get_downsampled_portions(
        left_point, right_point, original_length, desired_length
    )

    left_x_map_1d = get_scaled_array(left_point, left_len, sigma=sigma)
    right_x_map_1d = get_scaled_array(original_length - right_point, right_len, sigma=sigma)

    right_range = right_point + right_x_map_1d
    left_range = left_point - np.flip(left_x_map_1d)
    res = np.concatenate([left_range, np.arange(left_point, right_point, 1), right_range])
    return res.astype(np.float32)


def warp_no_crop(img, bbox: Bbox, total_desired_length, sigma=60):
    # Generate maps for both axes
    map_x_1d = _get_remap_array(bbox.x, bbox.x2, img.shape[1], total_desired_length, sigma)
    map_y_1d = _get_remap_array(bbox.y, bbox.y2, img.shape[0], total_desired_length, sigma)

    # Create the 2D grid
    map_x, map_y = np.meshgrid(map_x_1d, map_y_1d)

    # Remap - this pulls the WHOLE 1024 image into the 224 canvas
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)


# (get_scaled_array(1024, 100, ampl=10.0, sigma=30))

In [ ]:
sample = load_crop_level_sample_from_directory(
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/other/d3e42811-0d2e-485a-88fb-79bbfcef5214_75"
    )
)

In [ ]:
from mtrain.neg_mask.crops import bbox_only_mask
img = plt.imread("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall/1430787228341750/image.jpg")
mask = plt.imread("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall/1430787228341750/m2-md.png")
bb = Bbox(300, 300, 100, 100)
mask = bbox_only_mask(mask, bb, 2000)

show([img, mask, OV(img,mask)], (20,20), 3)

In [ ]:
from mtrain.neg_mask.crops import padded_bbox
# img,mask,bb= sample.full_image, sample.full_mask, sample.bbox
bb = padded_bbox(bb, 10, img.shape[:2])
rect = cv2.rectangle(img.copy(), (bb.x, bb.y), (bb.x2, bb.y2), [255,0,0], 2)

crop_img = img
crop_mask = mask
warped_mask = warp_no_crop(mask, bb, 224, sigma=100)
warped_img = warp_no_crop(img, bb, 224, sigma=40)

# show([warped_img, OV(warped_img, warped_mask)])

show([
    crop_img, warped_img,
    OV(crop_img, crop_mask), OV(warped_img, warped_mask),
])

foviate shrink is working now. ive got high expectations from it